<a href="https://colab.research.google.com/github/EstibenMT/machine_learning/blob/main/E3/Taller3_Ejercicio_Pr%C3%A1ctico_1_Predicci%C3%B3n_de_enfermedad_card%C3%ADaca.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Integrantes del equipo

Grupo: Jueves - Viernes 6:00 a 8:00 pm

- Estiben Montoya Taborda
- Luis David Lopera Parra
- Santiago Gomez Gallego
- Jose Miguel Buritica Morales

## Referencias

- Detrano, R., Janosi, A., Steinbrunn, W., Pfisterer, M., Schmid, J., Sandhu, S., Guppy, K., Lee, S., & Froelicher, V. (1989). *Heart Disease* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C52P4X
- Schlimmer, J. (1985). *Automobile* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5B01C

# Ejercicio Práctico 1: Predicción de enfermedad cardíaca

## Análisis clínico

A partir de un conjunto de registros clínicos se busca predecir el grado de enfermedad cardíaca que presenta una persona.

La predicción se realizará utilizando características como la edad, el sexo, la presión arterial, el colesterol, la frecuencia cardíaca máxima y otros resultados de exámenes médicos.

El resultado original se encuentra en la variable `num`, la cual toma 5 valores posibles. Siguiendo la indicación del docente, en este ejercicio **no se agrupan** estos valores en una variable binaria; se conservan las 5 categorías originales como una clasificación multiclase:

- `0`: ausencia de enfermedad (< 50% de estrechamiento de diámetro).
- `1`, `2`, `3`, `4`: distintos grados de presencia de enfermedad (> 50% de estrechamiento, en severidad creciente).

El modelo seleccionado será Logistic Regression, en su variante multinomial (soporta de forma nativa más de dos clases), debido a que el objetivo consiste en clasificar los registros entre las 5 categorías posibles de diagnóstico.

## Descripción de las variables seleccionadas

| Variable | Nombre | Tipo de dato | Tipo de variable | Descripción |
|---|---|---|---|---|
| Edad | `age` | Numérico | Numérica continua | Edad de la persona en años. |
| Sexo | `sex` | Numérico | Categórica binaria | Indica el sexo registrado para la persona. |
| Dolor torácico | `cp` | Numérico | Categórica nominal | Tipo de dolor en el pecho registrado en la evaluación clínica. |
| Presión arterial | `trestbps` | Numérico | Numérica continua | Presión arterial en reposo medida durante el examen. |
| Colesterol | `chol` | Numérico | Numérica continua | Nivel de colesterol sérico registrado para la persona. |
| Glucemia en ayunas | `fbs` | Numérico | Categórica binaria | Indica si la glucemia en ayunas es superior a 120 mg/dl. |
| Electrocardiograma | `restecg` | Numérico | Categórica nominal | Resultado del electrocardiograma en reposo. |
| Frecuencia cardíaca máxima | `thalach` | Numérico | Numérica continua | Frecuencia cardíaca máxima alcanzada durante la prueba. |
| Angina inducida | `exang` | Numérico | Categórica binaria | Indica si se presentó angina durante el ejercicio. |
| Depresión ST | `oldpeak` | Numérico | Numérica continua | Depresión del segmento ST inducida por el ejercicio. |
| Pendiente ST | `slope` | Numérico | Categórica ordinal | Pendiente del segmento ST durante el ejercicio máximo. |
| Vasos principales | `ca` | Numérico | Numérica discreta | Cantidad de vasos principales observados mediante fluoroscopia. |
| Talasemia | `thal` | Categórico | Categórica nominal | Resultado asociado con la prueba de talasemia. |
| Enfermedad cardíaca | `num` | Numérico | Variable objetivo binaria | Resultado que indica si la persona presenta o no enfermedad cardíaca. |

In [ ]:
!pip install ucimlrepo

from ucimlrepo import fetch_ucirepo

In [ ]:
# fetch dataset
heart_disease = fetch_ucirepo(id=45)

# data (as pandas dataframes)
X = heart_disease.data.features
y = heart_disease.data.targets

# metadata
print(heart_disease.metadata)

# variable information
print(heart_disease.variables)

{'uci_id': 45, 'name': 'Heart Disease', 'repository_url': 'https://archive.ics.uci.edu/dataset/45/heart+disease', 'data_url': 'https://archive.ics.uci.edu/static/public/45/data.csv', 'abstract': '4 databases: Cleveland, Hungary, Switzerland, and the VA Long Beach', 'area': 'Health and Medicine', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 303, 'num_features': 13, 'feature_types': ['Categorical', 'Integer', 'Real'], 'demographics': ['Age', 'Sex'], 'target_col': ['num'], 'index_col': None, 'has_missing_values': 'yes', 'missing_values_symbol': 'NaN', 'year_of_dataset_creation': 1989, 'last_updated': 'Fri Nov 03 2023', 'dataset_doi': '10.24432/C52P4X', 'creators': ['Andras Janosi', 'William Steinbrunn', 'Matthias Pfisterer', 'Robert Detrano'], 'intro_paper': {'ID': 231, 'type': 'NATIVE', 'title': 'International application of a new probability algorithm for the diagnosis of coronary artery disease.', 'authors': 'R. Detrano, A. Jánosi, W. Steinbrunn, M

### Información de las variables

En esta salida revisamos la información general de las variables del conjunto **Heart Disease**.

Se identifican **13 variables de entrada (`Feature`)** y la variable objetivo `num` (`Target`), que corresponde al diagnóstico de enfermedad cardíaca.

También podemos observar el tipo de cada variable, su descripción y si presenta valores faltantes. En este caso, UCI reporta datos faltantes en las variables `ca` y `thal`.

Esta revisión nos permite conocer la estructura de los datos antes de comenzar con la limpieza y el preprocesamiento.

## Importación de Pandas

Importamos Pandas porque ya vamos a empezar a revisar cómo vienen organizados los datos.

In [ ]:
import pandas as pd

## Preparación de los datos separados

En este ejercicio los datos ya vienen divididos en dos estructuras.

`X` contiene las variables clínicas que se utilizarán como entradas del modelo, mientras que `y` contiene la variable objetivo que representa el diagnóstico.

Trabajaremos directamente con estas dos estructuras, siguiendo la forma en que fueron entregados los datos.

In [ ]:
features_df = X.copy()
target_df = y.copy()

features_df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal
0,63,1,1,145,233,1,2,150,0,2.3,3,0.0,6.0
1,67,1,4,160,286,0,2,108,1,1.5,2,3.0,3.0
2,67,1,4,120,229,0,2,129,1,2.6,2,2.0,7.0
3,37,1,3,130,250,0,0,187,0,3.5,3,0.0,3.0
4,41,0,2,130,204,0,2,172,0,1.4,1,0.0,3.0


In [ ]:
target_df.head()

,num
0,0
1,2
2,1
3,0
4,0


### Revisión de las variables de entrada

Primero revisamos la estructura de `X` para conocer cuántos registros tiene, qué tipo de datos maneja y si existen valores faltantes.

In [ ]:
features_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 13 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       303 non-null    int64  
 1   sex       303 non-null    int64  
 2   cp        303 non-null    int64  
 3   trestbps  303 non-null    int64  
 4   chol      303 non-null    int64  
 5   fbs       303 non-null    int64  
 6   restecg   303 non-null    int64  
 7   thalach   303 non-null    int64  
 8   exang     303 non-null    int64  
 9   oldpeak   303 non-null    float64
 10  slope     303 non-null    int64  
 11  ca        299 non-null    float64
 12  thal      301 non-null    float64
dtypes: float64(3), int64(10)
memory usage: 30.9 KB


In [ ]:
features_df.isna().sum()

,0
age,0
sex,0
cp,0
trestbps,0
chol,0
fbs,0
restecg,0
thalach,0
exang,0
oldpeak,0


In [ ]:
features_df.duplicated().sum()

np.int64(0)

### Revisión de la variable objetivo

También revisamos `y`, porque la variable objetivo debe estar completa y correctamente organizada antes de entrenar el modelo.

In [ ]:
target_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   num     303 non-null    int64
dtypes: int64(1)
memory usage: 2.5 KB


In [ ]:
target_df.isna().sum()

,0
num,0


In [ ]:
target_df["num"].unique()

array([0, 2, 1, 3, 4])

In [ ]:
target_df["num"].value_counts().sort_index()

,count
num,
0,164
1,55
2,36
3,35
4,13


La documentación resumida de UCI describe la variable `num` utilizando los valores `0` y `1`. Sin embargo, al revisar directamente los registros descargados se observan 5 valores posibles: `0, 1, 2, 3, 4`.

Por instrucción del docente, para este ejercicio **no se agrupan** los valores mayores que cero en una sola clase. Se conservan las 5 categorías originales, y el problema se aborda como una **clasificación multiclase**: `0` corresponde a ausencia de enfermedad, y `1`, `2`, `3`, `4` representan distintos grados de severidad de la enfermedad.

Esto implica que la partición de los datos y la estratificación (`stratify`) deben realizarse sobre las 5 clases, no sobre una versión binaria, para que el conjunto de entrenamiento y el de prueba mantengan la misma proporción de cada una de las 5 categorías.

In [ ]:
#Aseguramos el tipo de dato entero para trabajar con valores discretos.
target_df["num"] = target_df["num"].astype(int)

### Distribución de las 5 categorías de la variable objetivo

Revisamos la proporción de cada una de las 5 categorías (`0` a `4`) antes de particionar los datos. Esta revisión es clave porque, a diferencia del enfoque binario, aquí hay clases muy minoritarias (por ejemplo, la clase `4` solo tiene 13 registros de 303), lo cual hace indispensable estratificar correctamente al dividir en entrenamiento y prueba.

In [ ]:
target_df["num"].value_counts(normalize=True)

,proportion
num,
0,0.541254
1,0.181518
2,0.118812
3,0.115512
4,0.042904


Observamos que las clases no están balanceadas: la clase 0 concentra el 54% de los registros, mientras que la clase 4 apenas representa el 4.3% (13 de 303 registros). Este desequilibrio entre 5 clases es muy marcado, es la razón principal por la que la estratificación en el train_test_split es obligatoria: sin stratify=y_model, existe el riesgo de que alguna de las clases minoritarias (como la 4) quede sub-representada o incluso ausente en el conjunto de prueba.

## Construcción de la matriz de características (X_model) y el vector objetivo (y_model)

Siguiendo el mismo patrón usado en los ejercicios de clase, definimos `X_model` como las variables de entrada y `y_model` como la variable objetivo, conservando sus 5 categorías originales (`0` a `4`) sin agrupar en formato binario.

In [ ]:
X_model = features_df.copy()
y_model = target_df["num"].copy()

X_model

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal
0,63,1,1,145,233,1,2,150,0,2.3,3,0.0,6.0
1,67,1,4,160,286,0,2,108,1,1.5,2,3.0,3.0
2,67,1,4,120,229,0,2,129,1,2.6,2,2.0,7.0
3,37,1,3,130,250,0,0,187,0,3.5,3,0.0,3.0
4,41,0,2,130,204,0,2,172,0,1.4,1,0.0,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
298,45,1,1,110,264,0,0,132,0,1.2,2,0.0,7.0
299,68,1,4,144,193,1,0,141,0,3.4,2,2.0,7.0
300,57,1,4,130,131,0,0,115,1,1.2,2,1.0,7.0
301,57,0,2,130,236,0,2,174,0,0.0,2,1.0,3.0


In [ ]:
y_model

,num
0,0
1,2
2,1
3,0
4,0
...,...
298,1
299,2
300,3
301,1


## Partición en conjuntos de entrenamiento y prueba

Dividimos los datos en un conjunto de entrenamiento (80%) y uno de prueba (20%), utilizando `stratify=y_model`. Como se diagnosticó en el paso anterior, las 4 clases minoritarias (`1, 2, 3, 4`) presentan desbalance severo (< 0.20), por lo que estratificar es obligatorio según la norma vista en clase: sin `stratify=y_model`, `train_test_split` dividiría los datos de forma completamente aleatoria y podría dejar muy pocos (o ningún) registro de las clases minoritarias en el conjunto de prueba, generando una evaluación sesgada del modelo.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_model, y_model, test_size=0.2, random_state=917, stratify=y_model)

In [ ]:
X_train

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal
291,55,0,2,132,342,0,0,166,0,1.2,1,0.0,3.0
6,62,0,4,140,268,0,2,160,0,3.6,3,2.0,3.0
229,66,1,4,112,212,0,2,132,1,0.1,1,1.0,3.0
280,57,1,4,110,335,0,0,143,1,3.0,2,1.0,7.0
218,64,0,4,130,303,0,0,122,0,2.0,2,2.0,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
88,53,0,4,138,234,0,2,160,0,0.0,1,0.0,3.0
97,60,0,4,150,258,0,2,157,0,2.6,2,2.0,7.0
9,53,1,4,140,203,1,2,155,1,3.1,3,0.0,7.0
76,60,1,4,125,258,0,2,141,1,2.8,2,1.0,7.0


## Verificación del tipo de dato de cada columna

Antes de construir el pipeline de preprocesamiento, verificamos con `dtypes` qué tipo de dato tiene cada columna. **Las columnas de tipo `object` (texto) requieren `OneHotEncoder`**; las columnas numéricas (`int64`/`float64`) se procesan con imputación y escalado.

In [ ]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 242 entries, 291 to 69
Data columns (total 13 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       242 non-null    int64  
 1   sex       242 non-null    int64  
 2   cp        242 non-null    int64  
 3   trestbps  242 non-null    int64  
 4   chol      242 non-null    int64  
 5   fbs       242 non-null    int64  
 6   restecg   242 non-null    int64  
 7   thalach   242 non-null    int64  
 8   exang     242 non-null    int64  
 9   oldpeak   242 non-null    float64
 10  slope     242 non-null    int64  
 11  ca        238 non-null    float64
 12  thal      240 non-null    float64
dtypes: float64(3), int64(10)
memory usage: 26.5 KB



### Conclusión: pipeline único

Como se observa en el paso anterior, **ninguna columna de `X_model` es de tipo `object`**: todas son `int64` o `float64`. Por lo tanto, siguiendo Se utiliza un **único pipeline** que aplica imputación y escalado a todas las columnas por igual.

Se usa SimpleImputer(strategy='median') porque ca y thal tienen valores faltantes y la mediana es robusta a valores atípicos, seguido de StandardScaler, necesario para que Logistic Regression funcione correctamente con variables en distintas escalas (por ejemplo, chol llega a valores de cientos, mientras que oldpeak está entre 0 y 6).

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

heart_pipeline = Pipeline(steps=[('imputacion', SimpleImputer(strategy='median')),('escalado', StandardScaler()),('clasificador', LogisticRegression(random_state=917, max_iter=1000))])

## Entrenamiento del pipeline con Logistic Regression

El modelo seleccionado es Logistic Regression. Al tener la variable objetivo 5 categorías (`0` a `4`), scikit-learn ajusta automáticamente una regresión logística **multinomial** (softmax), que estima la probabilidad de pertenecer a cada una de las 5 clases en lugar de una sola probabilidad binaria. Se incrementa `max_iter` respecto al ejercicio binario porque el optimizador multiclase suele requerir más iteraciones para converger.

In [ ]:
heart_pipeline.fit(X_train, y_train)

Pipeline(steps=[('imputacion', SimpleImputer(strategy='median')),
                ('escalado', StandardScaler()),
                ('clasificador', LogisticRegression(random_state=917))])

## Predicción y evaluación del modelo

Generamos las predicciones sobre el conjunto de prueba y calculamos la exactitud (`accuracy_score`) para evaluar el desempeño del modelo.

In [ ]:
y_pred = heart_pipeline.predict(X_test)

y_pred

array([3, 3, 3, 0, 0, 0, 0, 3, 3, 1, 0, 0, 0, 3, 0, 0, 0, 2, 0, 1, 0, 0,
       0, 3, 2, 0, 1, 0, 0, 2, 3, 0, 0, 3, 0, 0, 0, 2, 0, 0, 3, 1, 2, 0,
       0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 3, 2, 0, 0])

In [ ]:
from sklearn.metrics import accuracy_score

exactitud = accuracy_score(y_test, y_pred)
exactitud

0.5901639344262295

### Interpretación del resultado

El modelo obtuvo una exactitud de **0,59**, equivalente al **59 %** de aciertos en el conjunto de prueba.

En este ejercicio no estamos trabajando con dos clases, sino con cinco categorías de diagnóstico (`0` a `4`). Por lo tanto, el modelo debe distinguir entre cinco posibles resultados.

La exactitud obtenida está por encima de una predicción completamente aleatoria, que tendría aproximadamente un 20 % de probabilidad de acertar. Sin embargo, el resultado debe interpretarse con cuidado porque las clases están desbalanceadas:

- La clase `0` representa aproximadamente el 54,13 % de los registros.
- La clase `1` representa el 18,15 %.
- La clase `2` representa el 11,88 %.
- La clase `3` representa el 11,55 %.
- La clase `4` representa el 4,29 %.

La clase `4` tiene pocos registros, lo que dificulta que el modelo aprenda a reconocerla correctamente. Además, las clases `1`, `2`, `3` y `4` representan diferentes grados de enfermedad, por lo que pueden compartir características clínicas y resultar difíciles de diferenciar.

En conclusión, el modelo logró aprender algunos patrones de los datos y obtuvo un desempeño moderado. Sin embargo, la exactitud del 59 % no significa que reconozca correctamente todas las categorías por igual.

## Declaración sobre uso de IA Generativa

Se utilizó una herramienta de IA Generativa (Claude) como apoyo para:

- Redactar las celdas de Markdown explicativas y su formato.
- Consultar sobre el detalle de los métodos utilizados y más información de las librerías utilizadas.

**Prompts utilizados (resumen):**

1. "Explícame qué hace `train_test_split(X_model, y_model, test_size=0.2, random_state=917, stratify=y_model)`."
2. "¿Para qué sirve `SimpleImputer`, `StandardScaler` y `OneHotEncoder` dentro de un `Pipeline` de scikit-learn?"
3. "Ayúdame a redactar las celdas de Markdown explicando cada paso del flujo (partición, preprocesamiento, entrenamiento, evaluación) siguiendo el mismo formato usado en los ejercicios de clase."

Todo el código fue revisado, ejecutado y validado por el equipo antes de la entrega, y las interpretaciones de los resultados fueron completadas por el equipo con base en los valores obtenidos.